# 19 Bolshoi halos from CosmoSim

Halo positions for Bolshoi (250 h⁻¹ Mpc, 2048³ particles, WMAP5), so that the tracer-based void finders of notebook 18 (VIDE, VoidFinder) can run on it. The repo has Bolshoi's dark-matter density, T-web labels and halo grids (640³, 0.39 h⁻¹ Mpc), but not the halo positions behind the grids.

1. **Download** the most massive `N_KEEP` BDM halos (Bound Density Maximum, overdensity 360 × background) at z = 0 (snapshot 416) from CosmoSim's `bolshoi.bdmv` table through its TAP service, in one asynchronous job, and save them to Drive as Parquet. λ = 3 h⁻¹ Mpc needs (250/3)³ ≈ 579,000 halos; `N_KEEP = 2,000,000` leaves room for a hosts-only sample and stays under the service's 5,000,000-row limit.
2. **Where Bolshoi's grid points sit.** As for SMDPL (notebook 16): the λ = 3 halos are gridded on 640³ with point *i* at *i* × cell or at (*i* + ½) × cell, with every axis order, and correlated with Bolshoi's dark-matter density. The best placement goes into notebook 18 as `GRID_SHIFT`.
3. **Which halos the old grids used.** The repo's halo grids (`bolshoi/halos/L{3,5,7,10}`, from the dissertation) were made from BDM halos by code that is not in the repo. Our selection (most massive halos at each spacing; hosts and subhalos, or hosts only; mass-weighted or not) is gridded the same way and correlated with each old grid. A correlation near 1 means the same halos.

Results go to `data/Bolshoi/bolshoi_halo_check.json`, which notebook 18 reads. No GPU needed.

**Token.** As notebook 15: the CosmoSim API token in the Colab secret `COSMOSIM_TOKEN`; never written into the notebook.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install pyvo
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

In [ ]:
import dataclasses
import itertools
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyvo
import requests

from deepvoid import Catalog
from deepvoid.grid import cic
from deepvoid.observe import select_by_spacing
from deepvoid.sources.bolshoi import BOX_SIZE, SNAPSHOT_Z0, read_halos

catalog = Catalog()
DATA = catalog.root / 'Bolshoi'
HALO_FILE = DATA / 'bolshoi_bdmv_z0_top2M.parquet'
CHECK_FILE = DATA / 'bolshoi_halo_check.json'
N_KEEP = 2_000_000
SPACINGS = [3, 5, 7, 10]
TAP_URL = 'https://www.cosmosim.org/tap'
QUEUE = '1h'

session = requests.Session()
session.headers['Authorization'] = f"Token {userdata.get('COSMOSIM_TOKEN')}"
tap = pyvo.dal.TAPService(TAP_URL, session=session)
langs = []
for lang in tap.get_tap_capability().languages:
    versions = [v.content for v in lang.versions if v.content] if lang.versions else []
    langs += [f'{lang.name}-{v}' for v in versions] or [lang.name]
LANGUAGE = next((l for l in langs if l.lower().startswith('postgres')), None)
assert LANGUAGE, f'no PostgreSQL language listed: {langs}'
print('using', LANGUAGE)

COLUMNS = 'bdmid, hostflag, x, y, z, vx, vy, vz, mvir'
test = tap.run_sync(f'SELECT {COLUMNS} FROM bolshoi.bdmv WHERE snapnum = {SNAPSHOT_Z0} ORDER BY mvir DESC LIMIT 3',
                    language=LANGUAGE).to_table()
print(test)

## 1. Download

One job on the 1-hour queue: the `N_KEEP` most massive z = 0 halos. Skipped if the file is already on Drive.

In [ ]:
if HALO_FILE.exists():
    print(HALO_FILE.name, 'already there')
else:
    t0 = time.time()
    query = f'SELECT {COLUMNS} FROM bolshoi.bdmv WHERE snapnum = {SNAPSHOT_Z0} ORDER BY mvir DESC LIMIT {N_KEEP}'
    job = tap.submit_job(query, language=LANGUAGE, queue=QUEUE, runid='bolshoi_bdmv_z0_top')
    job.run()
    while job.phase not in ('COMPLETED', 'ERROR', 'ABORTED'):
        print(f'{job.phase} after {time.time() - t0:.0f} s', flush=True)
        time.sleep(30)
    if job.phase != 'COMPLETED':
        job.raise_if_error()
        raise RuntimeError(f'job ended as {job.phase}')
    df = job.fetch_result().to_table().to_pandas()
    print(f'{len(df):,} rows in {time.time() - t0:.0f} s')
    assert len(df) == N_KEEP, 'fewer halos than asked for: check the query'
    df = df.astype({'x': 'float32', 'y': 'float32', 'z': 'float32',
                    'vx': 'float32', 'vy': 'float32', 'vz': 'float32', 'mvir': 'float32'})
    DATA.mkdir(parents=True, exist_ok=True)
    df.to_parquet(HALO_FILE, index=False)
    job.delete()
    del df

halos = read_halos(HALO_FILE)
hosts = read_halos(HALO_FILE, distinct_only=True)
print(f'{len(halos):,} halos, {halos.meta["n_subhalos"]:,} of them subhalos; '
      f'mvir {halos.mass.min():.3g} to {halos.mass.max():.3g} h^-1 Msun; '
      f'positions {halos.pos.min():.2f} to {halos.pos.max():.2f} h^-1 Mpc')
for s in SPACINGS:
    need = round((BOX_SIZE / s) ** 3)
    print(f'λ = {s}: needs {need:,} halos; mass cut {np.sort(halos.mass)[-need]:.3g} (all), '
          f'{np.sort(hosts.mass)[-need]:.3g} (hosts only)')
assert len(hosts) >= round((BOX_SIZE / min(SPACINGS)) ** 3)

## 2. Where Bolshoi's grid points sit

Correlation of log(1 + ρ/ρ̄) between Bolshoi's dark-matter grid and the λ = 3 halos gridded with each placement (shift 0: point *i* at *i* × cell; shift ½: at (*i* + ½) × cell) and each axis order. The best should stand out and keep the axes as stored.

In [ ]:
dm = catalog.load('bolshoi/dm').data
n = dm.shape[0]
cell = BOX_SIZE / n
logd = np.log1p(dm / dm.mean()).ravel()
logd = (logd - logd.mean()) / logd.std()
del dm


def grid(tracers, shift, weights='mass'):
    # point i at (i + shift) * cell: move the tracers by -shift cells, then cic (point i at i * cell)
    moved = dataclasses.replace(tracers, pos=(tracers.pos - shift * cell) % BOX_SIZE)
    g = cic(moved, n, weights=weights).data
    return g / g.mean()


def corr(a, b):
    a = np.log1p(a).ravel(); a = (a - a.mean()) / a.std()
    return float(np.mean(a * b))


sample = select_by_spacing(halos, 3)
rows = []
for shift in (0.0, 0.5):
    g = grid(sample, shift)
    for perm in itertools.permutations(range(3)):
        rows.append({'shift': shift, 'axes': ''.join('xyz'[p] for p in perm), 'corr': corr(np.transpose(g, perm), logd)})
    del g
align = pd.DataFrame(rows).sort_values('corr', ascending=False)
print(align.to_string(index=False))
best = align.iloc[0]
ALIGNED = best['corr'] > 0.3
if ALIGNED:
    assert best['axes'] == 'xyz', 'the density matches best with its axes reordered; look before going on'
GRID_SHIFT = float(best['shift'])
other = align[(align['axes'] == 'xyz') & (align['shift'] != GRID_SHIFT)]['corr'].iloc[0]
print(f'GRID_SHIFT = {GRID_SHIFT} (correlation {best["corr"]:.4f} against {other:.4f})')
if not ALIGNED:
    print('NOT ALIGNED: the halos do not follow the dark matter for any placement or axis order; '
          'section 2b looks for the cause. Nothing is saved for notebook 18 until this passes.')
del logd, sample

### 2b Looking for the cause (runs only if section 2 failed)

If the halos do not follow the density, something maps their positions differently from the grid. On coarse 128³ grids (fast), this tries every combination of: a scale on the positions (1; 1/h and h for Mpc against h⁻¹ Mpc; 1/1000 for kpc), flipping each axis, and the 6 axis orders, and lists the best. The positions as stored are summarized first. A combination well above the rest (correlation of 0.5 or more) is the answer.

In [ ]:
from deepvoid.resample import block_mean
if not ALIGNED:
    p = halos.pos
    print('positions: min', p.min(0).round(3), 'max', p.max(0).round(3), 'mean', p.mean(0).round(3))
    raw = pd.read_parquet(HALO_FILE, columns=['x', 'y', 'z'])
    print('raw x, y, z before wrapping: min', raw.min().round(3).tolist(), 'max', raw.max().round(3).tolist())
    del raw
    nc = min(128, n)
    dmc = block_mean(catalog.load('bolshoi/dm').data.astype(np.float32), n // nc)
    lc = np.log1p(dmc / dmc.mean()).ravel(); lc = (lc - lc.mean()) / lc.std()
    del dmc
    sample = select_by_spacing(halos, 3)
    rows = []
    for scale in (1.0, 1 / 0.70, 0.70, 1e-3):
        moved = dataclasses.replace(sample, pos=(sample.pos * scale) % BOX_SIZE)
        g = cic(moved, nc, weights=None).data
        g = np.log1p(g / g.mean())
        for flips in itertools.product((False, True), repeat=3):
            gf = np.flip(g, [a for a in range(3) if flips[a]]) if any(flips) else g
            for perm in itertools.permutations(range(3)):
                v = np.transpose(gf, perm).ravel()
                rows.append({'scale': round(scale, 4), 'flip': ''.join('xyz'[a] for a in range(3) if flips[a]) or '-',
                             'axes': ''.join('xyz'[q] for q in perm),
                             'corr': float(np.mean((v - v.mean()) / v.std() * lc))})
    search = pd.DataFrame(rows).sort_values('corr', ascending=False)
    print(search.head(12).to_string(index=False))

## 3. Which halos the old grids used

For each old grid `bolshoi/halos/L{s}` and each of our selections, the correlation of log(1 + ρ/ρ̄) at the placement found above. Also the ratio of the old grid's total to the number of halos and to their total mass, which hints at whether it was weighted by mass.

In [ ]:
assert ALIGNED, 'section 2 failed: see section 2b before going on'
rows = []
for s in SPACINGS:
    old = catalog.load(f'bolshoi/halos/L{s}').data
    assert old.shape[0] == n
    o = np.log1p(old / old.mean()).ravel(); o = (o - o.mean()) / o.std()
    for label, tr in (('all', halos), ('hosts only', hosts)):
        sel = select_by_spacing(tr, s)
        for weights in ('mass', None):
            rows.append({'λ': s, 'halos': label, 'weights': weights or 'count',
                         'corr': corr(grid(sel, GRID_SHIFT, weights), o),
                         'old total / N': float(old.sum()) / len(sel.mass),
                         'old total / mass': float(old.sum()) / float(sel.mass.sum())})
    del old, o
match = pd.DataFrame(rows)
print(match.round(4).to_string(index=False))
best = match.groupby(['halos', 'weights'])['corr'].mean().idxmax()
print('best on average:', best)
check = {'aligned': bool(ALIGNED), 'alignment_corr': float(align.iloc[0]['corr']), 'grid_shift': GRID_SHIFT, 'distinct_only': best[0] == 'hosts only', 'weights': best[1],
         'halo_file': HALO_FILE.name, 'alignment': align.to_dict('records'), 'old_grid_match': match.to_dict('records')}
CHECK_FILE.write_text(json.dumps(check, indent=1))
print('saved', CHECK_FILE)

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive; anything kept only under `/content` goes with the runtime. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    import time
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime